In [ ]:
import polars as pl
df = pl.read_csv(
    source = "/home/hungpham/DE_Project_Exam/platforms/ingestion/source_data/de_assessment_data.csv"
)

In [ ]:
df.head()


In [ ]:
df.describe()

In [ ]:
df["event_type"].unique()

In [ ]:
df["rate_type"].unique()

In [ ]:
df["payment_method"].unique()

In [ ]:
df["vendor_id"].unique()

Với các thông tin này thì có thể thấy đây là thông tin về giao hàng hoặc chở khách,
Mỗi hàng là một chuyến đi (trip event) đã hoàn thành trong Q4 2024, là 1 dòng grant dữ liệu.
Đây là dữ liệu append-only fact — một event đã xảy ra thì không bao giờ thay đổi về mặt nghiệp vụ. 
Nhận diện dựa trên: zone_id, destination_id, passenger_count, duration, vendor_id, entity_id (driver/vehicle).

Từ dòng grant này thì sẽ có thể tách ra thành các bảng như sau:
1 là entity_table: bảng này sẽ mô tả các thông tin về tài xế thì sẽ có các cột như sau:
entity_id là mã tài xế, Primary key
rate_type   là đánh giá xem tài xế thuộc 5 sao, 4 sao, 3 sao, 2 sao hay 1 sao, tùy theo định nghĩa đánh giá của công ty/khách hàng về tài xế => foreign key
zone_id   là mã khu vực hoạt động, là foreign key/primary key trong bảng location
valid_from là tài xế này bắt đầu làm việc từ lúc nào, tùy thuộc vào quy định nghiệp vụ
valid_to   là thời hạn chót đến ngày tài xế hết hợp đồng, tùy theo quy định nghiệp vụ
is_current   là flag để có thể xác định tài xế còn hoạt động hay không,
 được dùng cho các mục đính kinh doanh, phân tích báo cáo, vì khi 
cập nhật thông tin tài xế mới thì có nhiều dòng thì có flag này sẽ đánh dấu đâu là dòng chính xác
created_at    ngày khởi tạo dữ liệu

2 là location_table: 
destination_id :  mã định danh của vị trí, nơi ở hoặc đích đến => primary_key
zone_id: mã định danh của vùng địa lý => foreign key
destination_name: Tên của vị trí
is_destination_current: Vị trí, tuyến này còn hoạt động vận hành không
valid_from: ngày mà tuyến đi này có hiệu lực
valid_to: ngày mà tuyến đường này ngừng hoạt động
create_at: ngày tạo dữ liệu

3 là zone_table
zone_id: mã định danh của vùng địa lý - primary_key
zone_name: Tên của vùng địa lý, tùy thuộc vào cách chia của nước sở tại hoặc quy định nghiệp vụ khác.
destination_id: mã định danh của vị trí, nơi ở hoặc đích đến => foreign key
is_zone_current: flag vùng này còn hiệu lực về địa nghĩa khu vực kinh doanh không, ví dụ Trước là Bình Phước nhưng giờ thành Đồng Nai
valid_from: ngày mà khái niệm vùng khai thác này có hiệu lực
valid_to: ngày mà khai niệm vùng khai thác này ngừng có hiệu lực
create_at: ngày tạo dữ liệu


# 4 là rate, này thì nào thay đối thì update là dc, có 
rate_type: mã số đánh giá tài xế => primary_key 
rate_name: mô tả xem từng mã có ý nghĩa gì, để xem xét tăng lương, thưởng phạt,...
updated_at: update thông tin khi có thay đổi

# 4 là vendor_table, có 2 vendor nên là có thay đổi thì upsert là dc rồi
vendor_id: mã đối tác => primary_key
vendor_name: thông tin đối tác
create_at: ngày tạo 
updated_at: ngày update


#5 là event_type_table

event_type_id: mã loại event => primary_key
event_type: category của event: bulk, prenium,..
update_at: ngày update


# 6 là trip_table

event_id: mã định danh chuyến đi => PK
event_timestamp: thời gian phát sinh
entity_id :       reference sang bảng entity
destionation_id :     reference sang bảng destination
zone_id :      reference sang bảng zone
vendor_id :    reference sang bảng vendor
rate_type : reference sang bảng rate    
event_type_id :reference sang bảng event
payment_method : thông tin thanh toán
duration_secs    : thời gian chuyến đi
passenger_count  : số lượng khách hàng
fare_amount      : giá vé 
surcharge_amount : giá vé bổ sung thêm
total_amount     : tỗng giá trị thu được
_inserted_at     : thời điểm insert dòng dữ liệu
_batch_id        : mã số batch_id
